# Hash vs Tree V0 Benchmark

이 노트북은 현재 코드베이스의 `hash_vs_tree_v0` 벤치를 실행하고, CSV 결과를 합친 뒤, pandas/matplotlib로 논문용 그래프를 생성한다.

현재 핵심 비교는 다음 두 구조다.

- `paged_hash_chain_v1`: `hash(key) -> bucket -> HashLeafPage chain`
- `hash_btree`: `FosterBtree` ordered by `hash(key) || key`

기본 설정은 빠른 smoke run이다. 연구용 run은 `N_KEYS`, `LOOKUPS_PER_THREAD`, `THREADS`, `DISTRIBUTIONS` 값을 키우면 된다.

## Dependencies

pandas/matplotlib가 없으면 이 셀이 현재 notebook kernel에 설치한다.

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = ["pandas", "matplotlib"]
missing_packages = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
else:
    print("pandas/matplotlib already available")

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
import datetime as dt
import math
import re
import shlex
import subprocess
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd


def find_repo(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "Cargo.toml").exists() and (path / "src/bin/hash_vs_tree_v0.rs").exists():
            return path
    raise RuntimeError("Could not find lipah repo root from current notebook directory")


REPO = find_repo(Path.cwd().resolve())
RUN_ID = dt.datetime.now().strftime("run_%Y%m%d_%H%M%S")
RESULT_ROOT = REPO / "results" / "hash_vs_tree_v0" / RUN_ID
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.75,
    "xtick.major.width": 0.75,
    "ytick.major.width": 0.75,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "figure.dpi": 160,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

VARIANT_LABELS = {
    "paged_hash_chain": "PagedHash",
    "paged_hash_chain_v1": "PagedHash-V1",
    "hash_btree": "HashBTree",
}

VARIANT_STYLES = {
    "paged_hash_chain": {"color": "#666666", "marker": "o", "linestyle": "--"},
    "paged_hash_chain_v1": {"color": "#0072B2", "marker": "s", "linestyle": "-"},
    "hash_btree": {"color": "#D55E00", "marker": "^", "linestyle": "-"},
}

print(f"REPO={REPO}")
print(f"RESULT_ROOT={RESULT_ROOT}")

## Experiment Knobs

처음에는 작은 값으로 파이프라인을 확인한다. 이후 다음처럼 키우면 된다.

- smoke: `N_KEYS=5_000`, `THREADS=[1,2,4]`
- medium: `N_KEYS=100_000` or `1_000_000`
- paper-scale start: `N_KEYS=10_000_000`, `THREADS=[1,2,4,8,16]`

`USE_RELEASE=True`는 실제 측정에 더 적절하지만, 첫 컴파일 시간이 더 걸린다.

In [ ]:
USE_RELEASE = False

VARIANT = "all"  # all = paged_hash_chain, paged_hash_chain_v1, hash_btree
N_KEYS = 5_000
KEY_SIZE = 8
VALUE_SIZE = 8
BUCKETS = 512

LOOKUPS_PER_THREAD = 5_000
WARMUP_LOOKUPS_PER_THREAD = 500
THREADS = [1, 2, 4]

# Uniform은 theta 값이 결과 컬럼을 맞추기 위해 기록만 된다.
# Zipf sweep을 추가하려면 예: [("uniform", 0.8), ("zipf", 0.5), ("zipf", 0.8), ("zipf", 0.95)]
DISTRIBUTIONS = [("uniform", 0.8)]

print({
    "use_release": USE_RELEASE,
    "variant": VARIANT,
    "n_keys": N_KEYS,
    "threads": THREADS,
    "distributions": DISTRIBUTIONS,
})

In [ ]:
def bench_command(distribution: str, theta: float, threads: int):
    cmd = ["cargo", "run"]
    if USE_RELEASE:
        cmd.append("--release")
    cmd += [
        "--bin", "hash_vs_tree_v0",
        "--no-default-features",
        "--",
        "--variant", VARIANT,
        "--n-keys", str(N_KEYS),
        "--lookups-per-thread", str(LOOKUPS_PER_THREAD),
        "--warmup-lookups-per-thread", str(WARMUP_LOOKUPS_PER_THREAD),
        "--threads", str(threads),
        "--distribution", distribution,
        "--zipf-theta", str(theta),
        "--buckets", str(BUCKETS),
        "--value-size", str(VALUE_SIZE),
        "--key-size", str(KEY_SIZE),
    ]
    return cmd


def run_one(distribution: str, theta: float, threads: int) -> Path:
    stem = f"{distribution}_theta_{theta:.3f}_t{threads}".replace(".", "p")
    csv_path = RESULT_ROOT / f"{stem}.csv"
    log_path = RESULT_ROOT / f"{stem}.stderr.log"
    cmd = bench_command(distribution, theta, threads)
    print("$", " ".join(shlex.quote(part) for part in cmd))
    completed = subprocess.run(
        cmd,
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    csv_path.write_text(completed.stdout, encoding="utf-8")
    log_path.write_text(completed.stderr, encoding="utf-8")
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f"benchmark failed: {csv_path}")
    return csv_path


csv_files = []
for distribution, theta in DISTRIBUTIONS:
    for threads in THREADS:
        csv_files.append(run_one(distribution, theta, threads))

csv_files

In [ ]:
combined_path = RESULT_ROOT / "combined.csv"
df = pd.concat([pd.read_csv(path) for path in csv_files], ignore_index=True)

numeric_cols = [
    "n_keys",
    "key_size",
    "page_size",
    "zipf_theta",
    "threads",
    "measured_lookups",
    "build_sec",
    "duration_sec",
    "throughput_ops_sec",
    "avg_latency_ns",
    "p50_latency_ns",
    "p95_latency_ns",
    "p99_latency_ns",
    "bucket_count",
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col])

df["throughput_mops_sec"] = df["throughput_ops_sec"] / 1_000_000
df["avg_latency_us"] = df["avg_latency_ns"] / 1_000
df["p50_latency_us"] = df["p50_latency_ns"] / 1_000
df["p95_latency_us"] = df["p95_latency_ns"] / 1_000
df["p99_latency_us"] = df["p99_latency_ns"] / 1_000

df.to_csv(combined_path, index=False)
print(f"wrote {combined_path}")
print(f"rows={len(df)}")
display(df)

In [ ]:
summary_cols = [
    "variant",
    "distribution",
    "zipf_theta",
    "threads",
    "throughput_mops_sec",
    "avg_latency_us",
    "p95_latency_us",
    "p99_latency_us",
]

summary = df[summary_cols].sort_values(["distribution", "zipf_theta", "threads", "variant"])
display(
    summary.style.format({
        "zipf_theta": "{:.2f}",
        "throughput_mops_sec": "{:.3f}",
        "avg_latency_us": "{:.2f}",
        "p95_latency_us": "{:.2f}",
        "p99_latency_us": "{:.2f}",
    })
)

## Paper-Style Plots

아래 셀은 single-column paper figure 기준으로 PDF/SVG/PNG를 저장한다. SIGMOD/DB 논문 draft에 넣을 때는 PDF를 우선 사용하면 된다.

In [ ]:
PLOT_DIR = RESULT_ROOT / "paper_plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

SINGLE_COL_FIGSIZE = (3.35, 2.25)
DOUBLE_COL_FIGSIZE = (6.95, 2.35)

PLOT_METRICS = [
    ("throughput_mops_sec", "Throughput (Mops/s)", "throughput"),
    ("avg_latency_us", "Avg. latency (us)", "avg_latency"),
    ("p50_latency_us", "P50 latency (us)", "p50_latency"),
    ("p95_latency_us", "P95 latency (us)", "p95_latency"),
    ("p99_latency_us", "P99 latency (us)", "p99_latency"),
]


def safe_name(text: str) -> str:
    return re.sub(r"[^A-Za-z0-9_]+", "_", text).strip("_")


def style_axis(ax, x_label: str, y_label: str):
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.grid(axis="y", color="0.88", linewidth=0.6)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", which="major", pad=2)


def save_figure(fig, stem: str):
    paths = []
    for suffix in ["pdf", "svg", "png"]:
        path = PLOT_DIR / f"{stem}.{suffix}"
        if suffix == "png":
            fig.savefig(path, dpi=300)
        else:
            fig.savefig(path)
        paths.append(path)
    return paths


def plot_metric(data: pd.DataFrame, metric: str, y_label: str, stem_metric: str, x_col: str = "threads"):
    output_paths = []
    groups = data.groupby(["distribution", "zipf_theta"], sort=True)
    for (distribution, theta), group in groups:
        fig, ax = plt.subplots(figsize=SINGLE_COL_FIGSIZE)
        for variant, variant_df in group.groupby("variant", sort=True):
            variant_df = variant_df.sort_values(x_col)
            style = VARIANT_STYLES.get(variant, {"marker": "o", "linestyle": "-"})
            ax.plot(
                variant_df[x_col],
                variant_df[metric],
                label=VARIANT_LABELS.get(variant, variant),
                linewidth=1.6,
                markersize=4.2,
                markeredgewidth=0.8,
                **style,
            )

        style_axis(ax, x_col.replace("_", " ").title(), y_label)
        ax.set_xticks(sorted(group[x_col].unique()))
        ax.margins(x=0.05)
        ax.legend(frameon=False, loc="best", handlelength=2.0, borderaxespad=0.2)
        fig.tight_layout(pad=0.3)

        stem = safe_name(f"{stem_metric}_by_{x_col}_{distribution}_theta_{theta:.3f}")
        paths = save_figure(fig, stem)
        output_paths.extend(paths)
        display(Markdown(f"### {stem}"))
        display(fig)
        plt.close(fig)
    return output_paths


all_plot_paths = []
for metric, y_label, stem_metric in PLOT_METRICS:
    all_plot_paths.extend(plot_metric(df, metric, y_label, stem_metric, x_col="threads"))

print("Saved paper figures:")
for path in all_plot_paths:
    print(path)

## Next Sweep Ideas

현재 노트북에서 바로 바꿔볼 값들:

- `DISTRIBUTIONS = [("uniform", 0.8), ("zipf", 0.5), ("zipf", 0.8), ("zipf", 0.95)]`
- `THREADS = [1, 2, 4, 8, 16]`
- `USE_RELEASE = True` for real measurements.
- `BUCKETS` sweep은 별도 loop를 추가해서 `x_col="bucket_count"`로 plot하면 된다.

논문용 실험으로 가려면 다음 단계는 `PagedHashChainV1` overflow/load-factor stats를 benchmark CSV에 붙이는 것이다.